Note: Write your code in the code cells, and your responses in markdown. 
Run the entire script and display the outputs of your code. 

Due: **11:59PM Central Time on Friday, 12/05**. Upload both your code (.ipynb) and responses (html or pdf) to Canvas by then. 

In [ ]:
# Packages you might need: (pip install ... if you don't have them)
import pandas as pd # for data manipulation
import os  # for setting directory 
# os.chdir() # input your personal directory where the dataset is saved
import statsmodels.formula.api as smf # for OLS regressions
import numpy as np  # to work with arrays (vectors/matrices)
import matplotlib.pyplot as plt # for plots

# RD Chile
The data set `rd.csv` contains student level data for $112,008$ students who finished high school and were eligible to enter college. In the specific country where the data orginate (Chile), students write a standardized test at the end of high school, called the “PSU” test. Their scores on this test, plus high school GPA, determine which colleges they can get into. Students who score at least 475 points on the PSU test are also eligible for a loan from the government for college costs, while students who score less than 475 points cannot receive the loan. 

In this problem set we will use regression discontinuity methods to analyze the effect of the loan program on the probability of college entry.

The variables on the data set are:
- psu = PSU test score (ranges from 300 to 700; the scores are numbers like 300.0, 300.5, 301.0, 301.5, 302.0....) 
- over475 = 1 if PSU score is 475 or higher

- entercollege = 1 if student entered college

- hsgpa = high school GPA (scored from 0 to 70, 70 is “perfect”)

- privatehs = 1 if student went to a private high school

- hidad = 1 if father has more than a high school education

- himom = 1 if mother has more than a high school education

In [ ]:
# Load dataset (using pandas "pd")
rd = pd.read_csv("rd.csv")
print(rd.head(2))

In [ ]:
print(rd.groupby(['over475'])['psu'].describe())

## 1. Visualization
Contruct the mean values of entercollege, hsgpa, privatehs, hidad, and himom for each integer value of PSU (e.g., get the mean for scores from 300 to 300.99, and assign that to the “300” bucket; then get the mean for scores from 301 to 301.99 and assign that to the “301” bucket,...). 

Show plots of these mean values as a function of PSU. You should see a jump in entercollege at 475 points, but relatively smooth values of the other variables.

In [ ]:
rd['psu'] = rd['psu'].apply(lambda x: np.floor(x))

# use `groupby` to collapse data and summarize at integer score level: 
buckets = rd.groupby(['psu'])[['entercollege','hsgpa','privatehs','hidad','himom']].mean().reset_index(drop=False)
buckets.loc[buckets['psu'].between(473,477)]

## 2. Local Linear Regressions
Next you will fit “local linear” regressions using different “bandwidths” to check that the student background variables do not jump at 475 points. To do this, you will regress one of the student characteristic variables, $x$, on the following controls:
- constant 1
- $psu$ (floor of psu score in `buckets`)
- $Z= (psu\geq 475)$ (the dummy for having score $\geq$ 475 – this is our main instrumental variable)
- $w = (psu-475) \times over475$ (note it equals the running variable when $psu\geq 475$, and 0 otherwise)

If you fit the model $$ x = \lambda_0 + \lambda_1 Z + \lambda_2 psu + \lambda_3 w + \epsilon $$ the coefficient $\lambda_1$ will measure the jump in "x" at $psu=475$. The slope of the line to left is $\lambda_2$, and the slope to the right is $\lambda_2+\lambda_3$. 


In [ ]:
buckets['Z'] = 0+(buckets['psu']>=475)
buckets['w'] = buckets['Z'] * (buckets['psu']-475)

### (a) Estimate RD given a bandwidth=10
Using the “collapsed” data - `buckets` from part 1, which has 1 observation per integer value of psu, and **a bandwidth of 10 points** on each side of the 475 cutoff, fit model (1) for $x\in\{hsgpa,hidad,himom\}$.

(HINT: what this means is that you fit the regression model to the collapsed data for the subset of data with $465\leq psu \leq484$. This data set will have 10 observations on scores less than 475, and 10 observations on scores of 475 or higher)

### (b) Alternative Bandwidth
repeat part (a) using a bandwidth of 20 points. Do you find that the estimated jumps are similar for all three variables as with a bandwidth of 10?

## 3. First Stage 
In this part you will fit the first stage model for the endogenous variable $D=$ enter college. (We don't have a subsequent outcome $y$ variable, but it could be something like earnings at age 35). Consider the first stage model: $$D=\pi_{0}+\pi_{1}Z+\pi_{2}psu+\pi_{3}w+ \varepsilon$$

- Using the “collapsed” data  - `buckets` from part 1 and a bandwidth of 10 points on each side of the 475 cutoff, fit the model above. 
- For every bandwidth from 5 to 50, develop a graph to show the robustness of the estimate of $\pi_{1}$ 

## 4. Compliers in RD
In this part you will use the Abadie trick to estimate the characteristics of the compliers. For some characteristic x the “goofy regression” setup is: 
$$ Dx	=	b_{0}+b_{1}D+b_{2}psu+b_{3}w+v $$
$$ D	=	\pi_{0}+\pi_{1}Z+\pi_{2}psu+\pi_{3}w+\varepsilon $$

Notice that this looks like the setup in Lecture 15, except we have to control for $psu$ and $w$. The coefficient $b_{1}$ provides an estimate of the mean of characteristic $x$ for the compliers who enter college just when they become eligible for the loan. 

Using the “collapsed” data - `buckets` from part 1 and a bandwidth of 10 points on each side of the 475 cutoff, fit the 2sls system and estimate the means of $\{hsgpa,hidad,himom\}$ for the compliers. How do these compare to the means for all students with PSU's between 465 and 484?

In [ ]:
# 2SLS:
from linearmodels.iv import IV2SLS # for IV2SLS # see solution to PS4
# you could also run FS, and RF separately to get the same point estimate (but incorrect SE)

In [ ]:
# Try one: 
bandwidth = 10 
x = 'hsgpa'
buckets[f'D_{x}'] = buckets['entercollege'] * buckets[x]
model = IV2SLS.from_formula(f'D_{x} ~ 1+ [entercollege ~ Z] + psu + w',data = buckets.loc[buckets['psu'].between(475-bandwidth, 475+bandwidth-1)]).fit(cov_type='robust')
print(model.summary)
print(model.first_stage)